# Sesión 14 — Redes Recurrentes y Modelado Temporal
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo IV · Fundamentos de Deep Learning**

## Objetivos de aprendizaje

1. Derivar el forward pass de la RNN y la retropropagación a través del tiempo (BPTT).
2. Comprender el problema del gradiente que se desvanece en RNN y cómo lo resuelve LSTM.
3. Implementar una celda LSTM desde cero y verificarla contra `nn.LSTM` de PyTorch.
4. Aplicar LSTM bidireccionales a la detección de crisis epilépticas en EEG.
5. Comprender arquitecturas secuencia-a-secuencia y sus aplicaciones biomédicas.

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Hochreiter, S. & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation*, 9(8). — El artículo original de LSTM. |
| ★★★ | Goodfellow, I. et al. (2016). *Deep Learning*. Cap. 10 (RNN). Gratis: deeplearningbook.org |
| ★★☆ | Graves, A. (2012). *Supervised Sequence Labelling with Recurrent Neural Networks*. Springer. Cap. 2–3. |
| ★★☆ | Tsiouris, K.M. et al. (2018). A long short-term memory deep learning network for the prediction of epileptic seizures. *Computers in Biology and Medicine*, 99. |
| ★☆☆ | LSTM ilustrado de Colah: https://colah.github.io/posts/2015-08-Understanding-LSTMs/ |

## Parte 0 — Configuración

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import train_test_split

rng    = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
})
print(f'Dispositivo: {device}')

## Parte 1 — RNN simple: ecuaciones y gradientes que se desvanecen

La actualización del estado oculto de la RNN:
$$\mathbf{h}_t = \tanh(\mathbf{W}_{hh}\mathbf{h}_{t-1} + \mathbf{W}_{xh}\mathbf{x}_t + \mathbf{b}_h)$$

BPTT calcula los gradientes desenrollando a través del tiempo:
$$\frac{\partial L}{\partial \mathbf{h}_0} = \frac{\partial L}{\partial \mathbf{h}_T} \prod_{t=1}^{T} \frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}} = \frac{\partial L}{\partial \mathbf{h}_T} \prod_{t=1}^{T} \mathbf{W}_{hh}^\top \text{diag}(1-\mathbf{h}_t^2)$$

Si $\|\mathbf{W}_{hh}\| < 1$: los gradientes **se desvanecen**. Si $\|\mathbf{W}_{hh}\| > 1$: **explotan**.

In [ ]:
# Demostrar el gradiente que se desvanece en una RNN simple
def flujo_gradiente_rnn(T, escala_W_hh, n_hidden=16):
    """Simula la magnitud del gradiente a través de T pasos de tiempo."""
    normas_grad = []
    # Matriz de peso recurrente aleatoria, escalada
    rng_w = np.random.default_rng(0)
    W_hh  = rng_w.normal(0, escala_W_hh / np.sqrt(n_hidden), (n_hidden, n_hidden))
    grad  = np.ones(n_hidden)
    h     = np.zeros(n_hidden)

    for t in range(T):
        dtanh = 1 - np.tanh(h)**2   # derivada de tanh
        grad  = (W_hh.T @ (dtanh * grad))
        h     = np.tanh(W_hh @ h)   # forward
        normas_grad.append(np.linalg.norm(grad))

    return np.array(normas_grad)

T_max = 100
escalas = [0.5, 0.9, 1.0, 1.1, 1.5]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for escala in escalas:
    gnorms = flujo_gradiente_rnn(T_max, escala)
    axes[0].semilogy(gnorms, lw=2, label=f'‖W_hh‖ escala={escala}')

axes[0].set(xlabel='Pasos de tiempo hacia atrás', ylabel='‖gradiente‖ (log)',
            title='Gradientes que se desvanecen/explotan en RNN simple')
axes[0].legend(fontsize=8)
axes[0].axhline(1, color='black', ls='--', lw=1)

# Comparar el flujo de gradiente RNN vs LSTM (simulado)
T_range = np.arange(1, T_max+1)
rnn_grad_desvanece = 0.9**T_range
lstm_grad_aprox    = np.exp(-0.01 * T_range)   # decaimiento mucho más lento — autopista del cell state

axes[1].semilogy(T_range, rnn_grad_desvanece, 'b-', lw=2.5, label='RNN (tanh, escala=0.9)')
axes[1].semilogy(T_range, lstm_grad_aprox,    'r-', lw=2.5, label='LSTM (carrusel de error constante)')
axes[1].set(xlabel='Pasos de tiempo hacia atrás', ylabel='Magnitud del gradiente (log)',
            title='RNN vs LSTM: flujo de gradiente en secuencias largas')
axes[1].legend()

plt.tight_layout()
plt.show()
print('LSTM mitiga el gradiente que se desvanece mediante el cell state '
      '("carrusel de error constante").')

## Parte 2 — Celda LSTM desde cero

La LSTM introduce un **estado de celda** $\mathbf{c}_t$ y tres compuertas:

$$\mathbf{f}_t = \sigma(\mathbf{W}_f[\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f) \qquad \text{(compuerta de olvido)}$$
$$\mathbf{i}_t = \sigma(\mathbf{W}_i[\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_i) \qquad \text{(compuerta de entrada)}$$
$$\tilde{\mathbf{c}}_t = \tanh(\mathbf{W}_c[\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_c) \qquad \text{(candidato de celda)}$$
$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t \qquad \text{(actualización de celda)}$$
$$\mathbf{o}_t = \sigma(\mathbf{W}_o[\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o) \qquad \text{(compuerta de salida)}$$
$$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t)$$

In [ ]:
class LSTMCell_scratch(nn.Module):
    """Celda LSTM individual — implementación transparente, equivalente a nn.LSTMCell."""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        # Una capa lineal combinada para las 4 compuertas (eficiente)
        self.W_ih = nn.Linear(input_size,  4 * hidden_size, bias=True)
        self.W_hh = nn.Linear(hidden_size, 4 * hidden_size, bias=True)

    def forward(self, x, h_prev, c_prev):
        """
        x      : (batch, input_size)
        h_prev : (batch, hidden_size)
        c_prev : (batch, hidden_size)
        """
        gates = self.W_ih(x) + self.W_hh(h_prev)   # (batch, 4*H)
        H = self.hidden_size
        i = torch.sigmoid(gates[:, 0*H : 1*H])      # compuerta de entrada
        f = torch.sigmoid(gates[:, 1*H : 2*H])      # compuerta de olvido
        g = torch.tanh(   gates[:, 2*H : 3*H])      # candidato de celda
        o = torch.sigmoid(gates[:, 3*H : 4*H])      # compuerta de salida

        c = f * c_prev + i * g
        h = o * torch.tanh(c)
        return h, c

# Verificar contra nn.LSTMCell
batch, T_test, d_in, d_h = 8, 20, 10, 32
x_test = torch.randn(T_test, batch, d_in)

lstm_scratch = LSTMCell_scratch(d_in, d_h)
lstm_pt      = nn.LSTMCell(d_in, d_h)

# Copiar pesos
with torch.no_grad():
    lstm_pt.weight_ih.copy_(lstm_scratch.W_ih.weight)
    lstm_pt.weight_hh.copy_(lstm_scratch.W_hh.weight)
    lstm_pt.bias_ih.copy_(lstm_scratch.W_ih.bias)
    lstm_pt.bias_hh.zero_()

h_s, c_s = torch.zeros(batch, d_h), torch.zeros(batch, d_h)
h_p, c_p = torch.zeros(batch, d_h), torch.zeros(batch, d_h)

for t in range(T_test):
    h_s, c_s = lstm_scratch(x_test[t], h_s, c_s)
    h_p, c_p = lstm_pt(x_test[t], (h_p, c_p))

err_h = (h_s - h_p).abs().max().item()
err_c = (c_s - c_p).abs().max().item()
print(f'Implementación propia vs nn.LSTMCell — error absoluto máx: h={err_h:.2e}, c={err_c:.2e}')
print('✅ Correcto' if err_h < 1e-5 and err_c < 1e-5 else '❌ Revisar implementación')

## Parte 3 — Bi-LSTM para detección de crisis epilépticas en EEG

Las LSTM **bidireccionales** procesan la secuencia tanto hacia adelante como hacia atrás,
dando a cada instante de tiempo contexto del pasado y del futuro — ideal para el análisis
de EEG offline.

In [ ]:
# ── Simular detección de crisis epilépticas estilo CHB-MIT ───────────────────
# Entrada: ventanas de 4s @ 256Hz = 1024 muestras → 8 características espectrales
# Tarea: detección binaria de crisis por ventana

def simular_secuencia_eeg(n_ventanas=200, n_features=8, ventanas_crisis=None, seed=0):
    """Simula una secuencia de ventanas de características espectrales de EEG."""
    rng_l = np.random.default_rng(seed)
    X_seq = rng_l.normal(0, 1, (n_ventanas, n_features)).astype(np.float32)
    y_seq = np.zeros(n_ventanas, dtype=np.int64)

    # Simular episodios ictales: potencia elevada en delta/theta, alfa suprimida
    if ventanas_crisis is None:
        inicio = rng_l.integers(60, 120)
        dur    = rng_l.integers(20, 50)
        ventanas_crisis = list(range(inicio, min(inicio+dur, n_ventanas)))

    for w in ventanas_crisis:
        X_seq[w, :3] += rng_l.normal(2.5, 0.5, 3)   # pico delta/theta/alfa
        X_seq[w, 3:6] -= rng_l.normal(1.0, 0.3, 3)  # beta/gamma suprimidas
        y_seq[w] = 1

    return X_seq, y_seq, ventanas_crisis

# Generar múltiples grabaciones (sujetos)
n_sujetos_eeg = 20
n_ventanas_por = 180
seqs_X, seqs_y = [], []
for s in range(n_sujetos_eeg):
    Xs, ys, _ = simular_secuencia_eeg(n_ventanas=n_ventanas_por, seed=s)
    seqs_X.append(Xs)
    seqs_y.append(ys)

X_eeg_seq = np.stack(seqs_X)   # (n_sujetos, T, F)
y_eeg_seq = np.stack(seqs_y)   # (n_sujetos, T)

# Normalización por característica (a través del tiempo)
X_norm = (X_eeg_seq - X_eeg_seq.mean(axis=1, keepdims=True)) / \
          (X_eeg_seq.std(axis=1, keepdims=True) + 1e-8)

# División train/test por sujeto (estilo LOSO)
sujetos_test  = [0, 1, 2, 3]
sujetos_train = [s for s in range(n_sujetos_eeg) if s not in sujetos_test]

Xtr_seq = torch.tensor(X_norm[sujetos_train], dtype=torch.float32)  # (16, T, F)
ytr_seq = torch.tensor(y_eeg_seq[sujetos_train], dtype=torch.long)  # (16, T)
Xte_seq = torch.tensor(X_norm[sujetos_test],  dtype=torch.float32)  # (4, T, F)
yte_seq = torch.tensor(y_eeg_seq[sujetos_test], dtype=torch.long)   # (4, T)

print(f'Secuencias de entrenamiento: {Xtr_seq.shape}  Secuencias de test: {Xte_seq.shape}')
print(f'Prevalencia de crisis (train): {ytr_seq.float().mean():.3f}')

In [ ]:
class BiLSTM_Crisis(nn.Module):
    """LSTM bidireccional para detección de crisis por ventana."""
    def __init__(self, input_size=8, hidden_size=64, n_layers=2,
                 dropout=0.3, n_classes=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size * 2, n_classes)   # ×2 por bidireccional

    def forward(self, x):
        # x: (batch, T, características)
        out, _ = self.lstm(x)          # (batch, T, 2*H)
        out    = self.dropout(out)
        logits = self.fc(out)          # (batch, T, n_classes) — predicción por ventana
        return logits


# También definimos una GRU unidireccional para comparar
class GRU_Crisis(nn.Module):
    def __init__(self, input_size=8, hidden_size=64, n_layers=2, n_classes=2):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, n_layers,
                           batch_first=True, dropout=0.3)
        self.fc  = nn.Linear(hidden_size, n_classes)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out)


def entrenar_modelo_secuencial(model, Xtr, ytr, n_epochs=40, lr=1e-3):
    model = model.to(device)
    opt   = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    # Peso de clase para el desbalance
    peso_pos = (ytr == 0).sum().float() / (ytr == 1).sum().float()
    crit = nn.CrossEntropyLoss(
        weight=torch.tensor([1.0, peso_pos.item()]).to(device))

    Xtr_d = Xtr.to(device)
    ytr_d = ytr.to(device)
    losses = []

    for ep in range(n_epochs):
        model.train()
        opt.zero_grad()
        logits = model(Xtr_d)             # (B, T, C)
        loss   = crit(logits.reshape(-1, 2), ytr_d.reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        losses.append(loss.item())

    return losses


modelos_rnn = {
    'BiLSTM (2 capas)': BiLSTM_Crisis(n_layers=2),
    'GRU (2 capas)':    GRU_Crisis(n_layers=2),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
resultados_auroc = {}

for nombre, model in modelos_rnn.items():
    print(f'Entrenando {nombre}...')
    hist = entrenar_modelo_secuencial(model, Xtr_seq, ytr_seq, n_epochs=50)

    model.eval()
    with torch.no_grad():
        logits_te = model(Xte_seq.to(device)).cpu()  # (4, T, 2)
        proba_te  = F.softmax(logits_te, dim=2)[:, :, 1].numpy()  # (4, T)
        preds_te  = logits_te.argmax(2).numpy()

    y_true_flat = yte_seq.numpy().ravel()
    p_flat      = proba_te.ravel()
    auroc = roc_auc_score(y_true_flat, p_flat)
    resultados_auroc[nombre] = auroc

    axes[0].plot(hist, lw=2, label=f'{nombre} (AUROC={auroc:.3f})')

axes[0].set(xlabel='Época', ylabel='Pérdida de entrenamiento',
            title='Modelos RNN — curvas de entrenamiento')
axes[0].legend(fontsize=8)

# Visualizar predicciones en un sujeto de test
model_bilstm = modelos_rnn['BiLSTM (2 capas)']
model_bilstm.eval()
with torch.no_grad():
    proba_viz = F.softmax(
        model_bilstm(Xte_seq[0:1].to(device)), dim=2
    )[0, :, 1].cpu().numpy()

y_viz = yte_seq[0].numpy()
t_viz = np.arange(n_ventanas_por)

axes[1].fill_between(t_viz, y_viz, alpha=0.3, color='tomato', label='Crisis verdadera')
axes[1].plot(t_viz, proba_viz, 'b-', lw=2, label='P(crisis) BiLSTM')
axes[1].axhline(0.5, color='gray', ls='--', lw=1)
axes[1].set(xlabel='Índice de ventana (4s cada una)', ylabel='Probabilidad',
            title='Probabilidad de crisis por ventana — sujeto de test')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()
for nombre, auroc in resultados_auroc.items():
    print(f'{nombre}: AUROC = {auroc:.4f}')

## Parte 4 — CNN vs RNN vs híbrido para clasificación de bioseñales

In [ ]:
class CNN_LSTM_Hibrido(nn.Module):
    """
    La CNN extrae características locales, la LSTM captura la dinámica temporal.
    Entrada: (batch, T, F) → reorganizada a (batch*T, 1, F) para Conv1d.
    """
    def __init__(self, n_features=8, cnn_channels=32, lstm_hidden=64, n_classes=2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, cnn_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(cnn_channels, cnn_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(4)
        )
        self.lstm = nn.LSTM(cnn_channels*4, lstm_hidden,
                             batch_first=True, bidirectional=True)
        self.fc   = nn.Linear(lstm_hidden*2, n_classes)

    def forward(self, x):
        B, T, F = x.shape
        # Aplicar CNN a cada ventana independientemente
        x_cnn = x.reshape(B*T, 1, F)              # (B*T, 1, F)
        x_cnn = self.cnn(x_cnn)                    # (B*T, C, 4)
        x_cnn = x_cnn.reshape(B, T, -1)            # (B, T, C*4)
        out, _ = self.lstm(x_cnn)                  # (B, T, 2*H)
        return self.fc(out)                         # (B, T, clases)


modelos_hibridos = {
    'BiLSTM':      BiLSTM_Crisis(n_layers=2),
    'CNN-BiLSTM':  CNN_LSTM_Hibrido(),
}

auroc_cmp = {}
for nombre, model in modelos_hibridos.items():
    print(f'Entrenando {nombre}...')
    entrenar_modelo_secuencial(model, Xtr_seq, ytr_seq, n_epochs=50)
    model.eval()
    with torch.no_grad():
        p = F.softmax(model(Xte_seq.to(device)), dim=2)[:,:,1].cpu().numpy().ravel()
    auroc_cmp[nombre] = roc_auc_score(yte_seq.numpy().ravel(), p)
    n_p = sum(par.numel() for par in model.parameters())
    print(f'  AUROC={auroc_cmp[nombre]:.4f}  |  parámetros={n_p:,}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(list(auroc_cmp.keys()), list(auroc_cmp.values()),
       color=['steelblue','tomato','seagreen'][:len(auroc_cmp)], edgecolor='white')
ax.set(ylabel='AUROC', title='Detección de crisis: comparación de arquitecturas',
       ylim=(0.5, 1.0))
plt.tight_layout()
plt.show()

## ✏️ Ejercicios

1. **Truncamiento de BPTT.** Implementa BPTT truncado (desenrollar solo $k$ pasos en vez
   de la secuencia completa). Compara tiempo de entrenamiento y AUROC para
   $k \in \{5, 20, 50, \text{completo}\}$ en la tarea de detección de crisis.

2. **Conteo de parámetros GRU vs LSTM.** Demuestra analíticamente que una GRU tiene
   $\frac{3}{4}$ de los parámetros de una LSTM con el mismo tamaño oculto. Verifícalo
   numéricamente con `sum(p.numel() for p in model.parameters())`. ¿Cuándo preferirías GRU?

3. **Pooling por atención.** Añade una capa de pooling por atención aprendible después de
   la BiLSTM para agregar las características por ventana en una sola puntuación de
   riesgo de crisis a nivel de paciente. Compara el AUC por ventana vs el AUC a nivel
   de paciente.

4. **EEG real de CHB-MIT.** Descarga la base de datos CHB-MIT Scalp EEG
   (physionet.org/content/chbmit/). Extrae características de épocas de 1 segundo
   (potencias de banda delta/theta/alfa/beta/gamma) usando MNE-Python. Entrena la
   BiLSTM y reporta el AUROC con protocolo LOSO entre sujetos.

5. *(Desafío)* **Secuencia-a-secuencia para eliminación de ruido en ECG.** Construye un
   encoder-decoder LSTM que tome una secuencia de ECG ruidosa como entrada y produzca
   una versión limpia. Entrena con ruido de artefacto de movimiento simulado añadido a
   latidos de ritmo sinusal normal de PhysioNet. Evalúa con métricas de SNR y RMSE.

## 📚 Conjuntos de datos

| Conjunto de datos | Fuente | Notas |
|---|---|
| CHB-MIT Scalp EEG | https://physionet.org/content/chbmit/ | Epilepsia pediátrica, 23 sujetos |
| Temple University EEG Corpus | https://isip.piconepress.com/projects/tuh_eeg/ | El corpus público de EEG más grande |
| PhysioNet NSR DB | https://physionet.org/content/nsrdb/ | ECG de ritmo sinusal normal |